# HPT — the UI only

Google Colab → **Runtime → Change runtime type → A100** (an L4 works; it is slower per query).

No server is started for the model. `vllm.LLM(...)` loads MedGemma into this notebook's own
process and returns an object; generation is a method call, no port, no `vllm serve`.
The one instance built in step 4 backs both the chat page and the pipeline dashboard.

You need: Drive with `New_PT_DB_Backups/New_PT_DB.tar.gz` and
`Price_Transparency/datasets/everyUPMCmrf_clean (1).csv`, and a Hugging Face token
with access to the two gated models (set `HF_TOKEN` in Colab secrets or `.env`).

Order: code → data → model → UI.

## 1. The code (branch `no-router`)

In [ ]:
%cd /content
import os, subprocess

REPO, BRANCH = "/content/HPT_Prototype", "no-router"

if not os.path.isdir(os.path.join(REPO, ".git")):
    print(subprocess.run(
        ["git", "clone", "-b", BRANCH, "https://github.com/adi274903/HPT_Prototype", REPO],
        capture_output=True, text=True).stderr.strip())

# -f because a previous session may have left local edits that block the checkout;
# the remote branch is the only thing worth keeping in this runtime.
for cmd in (["git", "fetch", "--all", "--prune"],
            ["git", "checkout", "-f", BRANCH],
            ["git", "reset", "--hard", f"origin/{BRANCH}"]):
    out = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
    if out.returncode:
        print(f"FAILED {' '.join(cmd)}: {out.stderr.strip()}")

print(subprocess.run(["git", "log", "--oneline", "-1", "--decorate"],
                     cwd=REPO, capture_output=True, text=True).stdout.strip())

## 2. Dependencies

In [ ]:
%cd /content/HPT_Prototype
!pip install -q vllm duckdb sentence-transformers qdrant-client pandas python-dotenv
!python -c "import vllm, duckdb, sentence_transformers, qdrant_client; print('deps ok', vllm.__version__)"

If `import vllm` fails *below*, **Runtime → Restart session** and re-run from step 1
(the pip install stays cached; vLLM wants a clean CUDA context).

## 3. Drive, snapshot, MRF

In [ ]:
import os, sys, pathlib
import colab_run

colab_run.mount_drive()

paths = colab_run.default_paths()
DB = pathlib.Path(paths["qdrant_path"])          # /content/New_PT_DB

# stage_db() copies ~500 MB from Drive and untars it on every call; a warm runtime
# already has the store, so only extract when it is actually missing.
if (DB / "meta.json").exists():
    print(f"snapshot already staged: {DB}")
else:
    DB = pathlib.Path(colab_run.stage_db())

os.environ["PT_QDRANT_PATH"] = str(DB)           # setup() and the UI read this

for key, value in paths.items():
    print(f"{key:18} {value}   {'ok' if os.path.exists(value) else 'MISSING'}")

# The store must be readable and populated — and closed again: embedded Qdrant
# allows one client per directory, and setup() opens it next.
from qdrant_client import QdrantClient
client = QdrantClient(path=str(DB))
for name in ("cpt_medte", "hcpcs_medte"):
    print(f"{name:12} {client.count(name).count:,} points")
client.close()

## 4. One vLLM instance for the UI

In [ ]:
from vllm import LLM
from pt_healthcare.backends import VLLMPipe

SHARED = LLM(
    model="google/medgemma-1.5-4b-it",
    dtype="bfloat16",
    max_model_len=32768,
    gpu_memory_utilization=0.85,   # leaves room for the MedTE embedder beside it
)

gateway  = VLLMPipe("google/medgemma-1.5-4b-it", llm=SHARED)
pipeline = colab_run.setup(mount=False, stage=False, qdrant_path=str(DB), pipe=gateway)
print("pipeline ready")

MedGemma 1.5 under vLLM is the one piece that cannot be checked outside a GPU.
If this cell throws while loading, the transformers fallback works for every later
cell — just slower:

```python
pipeline = colab_run.setup(mount=False, stage=False, qdrant_path=str(DB))
```

Either way, **never** call `evaluate(backend="vllm")` or build a second `LLM` while
`SHARED` exists: two copies of the weights do not fit.

## 5. Serve the UI

In [ ]:
ui = colab_run.serve_ui(pipeline=pipeline, page="patient", tunnel="cloudflare")
print(ui.pages)
# ui.stop() when finished — the tunnel URL is public and unauthenticated

## 6. Check the speed, then keep it honest

In [ ]:
import time

QUERIES = [
    "What might a diagnostic mammogram cost at UPMC Presbyterian?",
    "Colonoscopy at UPMC Shadyside with UPMC Health Plan",
]

for query in QUERIES:
    started = time.perf_counter()
    result = pipeline.run(query)
    elapsed = time.perf_counter() - started
    print(f"{elapsed:5.1f}s  {len(result.get('answer','')):4d} chars  {query[:48]}")

Each step logs `(generated tokens, prompt tokens, decode tok/s)`. Read them before
changing anything:

- **prompt tokens high** → the answer context grew; check `build_answer_context` caps.
- **generated tokens near the budget** → the model is rambling. Budgets are
  *dynamic* now (`prompt_tokens * factor + slack`, floored and capped at
  `PT_MAX_NEW_TOKENS`), so `PT_MAX_NEW_TOKENS_<STEP>` (e.g. `_DECIDE=2048`)
  pins one step as an escape hatch rather than being the default.
- **tok/s low** → decode bound: a Q4_K_M GGUF alongside `llama_cpp` roughly doubles it,
  and at 4B the quality difference on these three short tasks is small.

## 7. Shut down

In [ ]:
ui.stop()          # server and tunnel
del SHARED         # frees the GPU without a runtime reset
print("stopped")